In [ ]:
import xarray as xr

# 1. read somo.ml

In [ ]:
dl=[]
for year in range(2001,2019):
    f='/Net/Groups/BGI/data/DataStructureMDI/DATA/grid/Global/0d25_daily/somo_ml/v1/Data/layer1/'+'layer1.somo_ml_v1.1440.720.'+str(year)+'.nc'
    d=xr.open_dataarray(f)
    dl.append(d)
sms=xr.concat(dl,dim='time')

# 2. use "mean" to coarsen data

In [ ]:
sms_coarsen=sms.coarsen(latitude=3,longitude=3).mean().sel(latitude=slice(79,-81))

# 3. deal with "-9999 error"

In [ ]:
sms_coarsen['longitude'] =sms_coarsen['longitude'].where(sms_coarsen['longitude'] != -9999)
sms_coarsen['latitude'] = sms_coarsen['latitude'].where(sms_coarsen['latitude'] != -9999)
# sms_coarsen.to_netcdf('/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/somo.nc')

# 4. detrend, deseasonality functions

In [ ]:
def detrend_dim(da, dim, deg=1):
    p = da.polyfit(dim=dim, deg=deg)
    fit = xr.polyval(da[dim], p.polyfit_coefficients)
    return da - fit

def deseasonality(var_dt):
    var_mean = var_dt.groupby('time.month').mean(dim='time')
    var_anom = var_dt.groupby('time.month') - var_mean
    return var_anom

# Main processing
sms_coarsen=sms.coarsen(latitude=3,longitude=3).mean().sel(latitude=slice(79,-81))
var = sms_coarsen.copy()
var_dt = detrend_dim(var, dim='time', deg=1)
var_anom = deseasonality(var_dt)
var_anom['longitude'] = var_anom['longitude'].where(var_anom['longitude'] != -9999)
var_anom['latitude'] = var_anom['latitude'].where(var_anom['latitude'] != -9999)